In [1]:
import gc

import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from default_risk.scripts.auxiliars_for_modeling import apply_cyclical_encoding
import joblib
import lightgbm as lgb
from typing import Optional, List



import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import get_pipeline

from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application")
#auc_score_OOF= 0.754

#freeing memory
del merged_df
gc.collect()



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)




#run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"prev_app+installment fpi",enable_feature_permutation=True)

#importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_rows_internal_parent.csv")
#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "max_rows.csv")
#X = clean_noise_from_feature_importance(importance_df,X,0.0025)
#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.0003)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"raw_internal_parent_target_encoding",persist_feature_importance=True)

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"raw_internal_parent_target_encoding",persist_feature_importance=True)

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "pipeline_baseline.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "max_cols_internal.csv")
importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_cols_internal.csv")

#X = clean_noise_from_feature_importance(importance_df,X,0.0025)
X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00010)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"max_cols_internal")

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

X= apply_cyclical_encoding(X,"hour_appr_process_start_prev_1",24)




X.drop(columns=["hour_appr_process_start_prev_1"],inplace=True)

model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type","code_reject_reason_prev_1","name_income_type","name_goods_category_prev_1","name_cash_loan_purpose_prev_1"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)#,,"product_combination_prev_1"

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_parent_target_enconding_max_cols_feature_importance.csv")
importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_internal_parent_target_enconding_max_cols.csv")

#X = clean_noise_from_feature_importance(importance_df,X,0.0024739875)
#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.0003)

#pd.get_dummies(X,columns= ["name_contract_type"])



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"internal_parent_target_enconding_max_cols")


#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()  

In [ ]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#now bureau parent (main with feature engineering + Bureau with feature engineering + Bureau_balance)
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "toxic_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)



#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X)
#importance_permutation_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")
importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "external_importance.csv")
#criteria = creating_criteria(importance_df,importance_permutation_df)

#importance2 = pd.read_csv(cfg.ARTIFACTS_DIR / "second filter.csv") #
#X= X.drop(columns=["bureau_balance_is_delincuency_sum_loan_1","bureau_has_bureau_balance_data_loan_1","ext_source_1_is_missing"]) 

X= clean_noise_from_feature_importance(importance_df,X,0.004)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"external_parent_co_sample_clean")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

model= xgb.XGBClassifier(**hiperparams)
categorical_features=  ["organization_type","occupation_type","name_income_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_external_parent_target_encoding.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X,0.0003)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"external_parent_target_encoding")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "final_importance.csv")

X= clean_noise_from_feature_importance(feature_raper,X)

merged_df= merged_df.drop(columns=["flag_email"])

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "pipeline_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)


#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")



merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()





X,Y = prepare_columns(merged_df)

X = cast_object_into_categoricals(X)

#importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_cols_internal.csv")




#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00010)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"max_rows_final_model")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

features_from_internal_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_best_result.csv")
features_from_external_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "external_best_result.csv")

internal_list=  features_from_internal_historial["feature_name"].to_list()
external_list=  features_from_external_historial["feature_name"].to_list()
features_names = list(set(internal_list + external_list))

X,Y = prepare_columns(merged_df)

X= X[features_names]

X= X.drop(columns= ["amt_down_payment_sum"])



X = cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"final_model_from_convination_of_best_results")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

par = {
"n_estimators" : 1000,
"learning_rate" : 0.03,
"num_leaves" :31,          
"max_depth" :-1,           
"subsample" : 0.8,          
"colsample_bytree" : 0.8,   
"random_state" : 42,
"n_jobs" : -1,
"objective" : 'binary',
"force_col_wise": True
}

model_lgbm = lgb.LGBMClassifier(**par)


categorical_features= ["organization_type","occupation_type","name_goods_category_prev_1","name_cash_loan_purpose_prev_1"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",


pipeline= get_pipeline(50,categorical_features,model_lgbm)





#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df,santize_text=True)



features_df= pd.read_csv(cfg.ARTIFACTS_DIR / "features_final_model.csv")

feature_list=  features_df["feature_name"].to_list()



feature_list = feature_list + ["amt_income_total"]#



X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

features_to_target_encoding= ["instalment_income_ratio"]

X = cast_object_into_categoricals(X)

model=xgb.XGBClassifier(**hiperparams)



run_cv_tracked_mlflow(pipeline,par,cv,X,Y,experiment_name,"lightgbm")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

[LightGBM] [Info] Number of positive: 19843, number of negative: 225933
[LightGBM] [Info] Total Bins 98531
[LightGBM] [Info] Number of data points in the train set: 245776, number of used features: 637
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080736 -> initscore=-2.432387
[LightGBM] [Info] Start training from score -2.432387


In [ ]:
def eliminar_colinealidad(X: pd.DataFrame, umbral: float = 0.95, metodo: str = 'pearson') -> pd.DataFrame:

    X_num = X.select_dtypes(include=[np.number])
    
    corr_matrix = X_num.corr(method=metodo).abs()
    
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
   
    columnas_a_eliminar = [columna for columna in upper_tri.columns if any(upper_tri[columna] > umbral)]
    
    # 5. Retornar el DataFrame original sin esas columnas
    return columnas_a_eliminar

In [ ]:
def clean_colineality(X: pd.DataFrame, column_list) -> pd.DataFrame:

    return X.drop(columns=column_list)

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()
list_to_delete= eliminar_colinealidad(merged_df,0.95)




In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)


model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)



#X= X.drop(columns=cols_to_drop)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"monster_without_co_lineality",enable_feature_permutation=False)

#auc_score_OOF=  0.781

In [3]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)




model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)

cols_to_drop= ["name_income_type"]

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")


X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)

X= X.drop(columns=cols_to_drop)






#five_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_fine_pruned_final_model.csv")

#X = clean_importance_zero_and_negative_pfi(five_filter,X,0.00009)

run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"fine_pruned_final_model")


eliminando ['diff_application_credit_median', 'building_score_mean', 'log_amt_application_std', 'closed_days_credit_update_closed_mean', 'credit_card_is_over_the_limit_sum_prev_1', 'active_days_credit_active_min', 'ratio_credit_to_annuity_prev_1', 'bureau_balance_amount_rows_with_activity_loan_1', 'closed_balance_months_balance_min_closed_min', 'instalments_repeated_for_underpayment_mean_prev_1', 'log_amt_credit_std', 'credit_card_amt_balance_std_prev_1', 'global_approval_ratio', 'log_amt_credit_mean', 'fondkapremont_mode', 'weekday_appr_process_start_sin', 'flag_live_city_not_work', 'active_amt_annuity_active_mean', 'instalments_log_amt_instalment_mean_prev_1', 'active_amt_annuity_active_std', 'amt_req_credit_breau_mon', 'instalments_raw_size_serie_prev_1', 'active_balance_months_balance_min_active_min', 'instalments_is_delinquency_sum_prev_1', 'building_score_std', 'log_diff_application_credit_prev_1', 'nflag_insured_on_approval_sum', 'active_amt_credit_sum_limit_active_std', 'bureau

(0.7858240747994559, 0.0032122314516403313)

In [ ]:
test_application= pd.read_parquet(cfg.MASTER_DATA_DIR / "prepared_dataset_test.parquet")
dtale.show(test_application.head(5))